# Train LeRobot ACT — OpenArm + RH56 grasp policy (Kaggle / Colab GPU)

1. **Kaggle**: *Add Input* → upload `openarm_grasp.zip` as a Dataset (it appears under `/kaggle/input/...`). Settings → Accelerator **GPU T4 x2 / P100**, Internet **on**. Then *Save Version → Save & Run All* (runs in the background, ~1 h).
2. **Colab**: Runtime → Change runtime type → **T4 GPU**; upload `openarm_grasp.zip` with the file panel; *Run all*.
3. Download `act_openarm.zip` from the output and unzip it into `artifacts/lerobot_runs/` on the laptop.

In [ ]:
!pip install -q "lerobot==0.4.0" "huggingface-hub==0.35.3" "datasets==4.1.1"
# Kaggle's preinstalled transformers needs a newer huggingface_hub than LeRobot 0.4 allows,
# and LeRobot imports it only for GR00T: remove it (the laptop's working env has none).
!pip uninstall -y -q transformers
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!python -c "import lerobot.scripts.lerobot_train; print('lerobot-train imports OK')"

In [ ]:
import glob, os, shutil, zipfile, pathlib
# Find the dataset: an uploaded zip (Colab), or the folder Kaggle extracted it into.
root = pathlib.Path('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
DATA = root / 'data' / 'openarm_grasp'
shutil.rmtree(DATA, ignore_errors=True)
zips = glob.glob('/kaggle/input/**/openarm_grasp.zip', recursive=True) + glob.glob('/content/openarm_grasp.zip')
if zips:
    zipfile.ZipFile(zips[0]).extractall(root / 'data')
else:
    info = glob.glob('/kaggle/input/**/meta/info.json', recursive=True)
    print('found:', info)
    shutil.copytree(pathlib.Path(info[0]).parent.parent, DATA)
OUT = root / 'act_openarm'
print(DATA, sorted(os.listdir(DATA)))

In [ ]:
# ACT, state-only (observation.state + observation.environment_state), 10 Hz,
# chunk 10 (1 s); temporal ensembling is set at inference time.
# 10k steps x batch 128 is ~75 passes over ~17k frames (40k was ~560: slow, and it
# overfits); AMP uses the T4's fp16 tensor cores; 4 loader workers = Kaggle's 4 CPUs.
!lerobot-train   --dataset.repo_id=local/openarm_grasp --dataset.root={DATA}   --policy.type=act --policy.device=cuda --policy.use_amp=true --policy.push_to_hub=false   --policy.chunk_size=10 --policy.n_action_steps=10   --output_dir={OUT} --steps=10000 --batch_size=128   --log_freq=500 --save_freq=10000 --eval_freq=0   --wandb.enable=false --num_workers=4
import os
assert os.path.isdir(OUT), 'lerobot-train produced no output: see the log above'

In [ ]:
import shutil
shutil.make_archive(str(root / 'act_openarm'), 'zip', OUT)
print('download:', root / 'act_openarm.zip')